In [ ]:
import torch
from tokenise import BPE_token
import os
import pandas as pd
from os.path import join
import numpy as np
import torch.nn as nn
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
from transformers import  LineByLineTextDataset
from ml_things import plot_dict, plot_confusion_matrix, fix_text
from sklearn.metrics import classification_report, accuracy_score
from transformers import (
                          GPT2Config,
                          GPT2Tokenizer,
                          AdamW,
                         )

## generate corpus

In [ ]:
from right_corpus import NP, NPVP, SOV
print(len(NP + NPVP + SOV))

In [ ]:

df_ma = pd.read_excel('./MATERIAL.xlsx')
id_2_dur=dict(zip(df_ma['TrialID'],df_ma['soundDur']))


trail_list = [ str(df_ma['word1'][i]) + ' ' + str(df_ma['word2'][i]) + ' '  + str(df_ma['word3'][i]) + ' ' + str(df_ma['word4'][i]) + ' '   
              + str(df_ma['word5'][i]) + ' '   + str(df_ma['word6'][i]) + ' '   + str(df_ma['word7'][i]) + ' '   + str(df_ma['word8'][i])
               for i in range(len(df_ma))]
trail_list = [sentence.replace("nan", "").strip() for sentence in trail_list]


def remove_duplicates(sentence):
    words = sentence.split()  
    for i in range(7):
        if words[-1] ==words[-2]:
            words.pop(-1)
    return ' '.join(words)  

new_list = [remove_duplicates(sentence) for sentence in trail_list]

id_2_trail_type = dict(zip(list(df_ma['TrialID']),new_list))

new_new_list = []
for item in new_list:
    if item not in new_new_list:
        new_new_list.append(item)



new_list_co = []
new_list_unc = []
for co in new_list:
    if co in NP + NPVP + SOV:
        new_list_co.append(co)
    else:
        new_list_unc.append(co)

new_list = new_list_co + new_list_unc


y_train = []
for itemt in new_list:
    if itemt in NP + NPVP + SOV:
        y_train.append('true')
    else:
        y_train.append('false')


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
y_train_tensor = []
for itemt in new_list:
    if itemt in NP + NPVP + SOV:
        y_train_tensor.append(torch.tensor(0).to(device))
    else:
        y_train_tensor.append(torch.tensor(1).to(device))

criterion = torch.nn.CrossEntropyLoss()

In [ ]:
import random

random.seed(42)
assert len(new_list) == len(y_train)

num_elements = int(0.8 * len(new_list))


random_indices = random.sample(range(len(new_list)), num_elements)


remaining_indices = list(set(range(len(new_list))) - set(random_indices))


train_list = [new_list[i] for i in random_indices]
train_labels = [y_train_tensor[i] for i in random_indices]
val_list = [new_list[i] for i in remaining_indices]
val_labels = [y_train_tensor[i] for i in remaining_indices]


## evaluate functions

In [ ]:
def evaluate_np_npvp_sov(model, tokenizer, test_dataset, per_for_train,temp):
    np_generated = []
    npvp_generated = []
    sov_generated = []
    model.eval()
    correct_count = 0
    total_generated = 0
    new = 0
    cor = []
    cor_new = []
    
    with open("./text/all_sentences.txt", "r", encoding="utf-8") as f:
        full_corpus = f.read().splitlines()
    with open(f"./text/train{per_for_train}.txt", "r", encoding="utf-8") as ff:
        train_corpus = ff.read().splitlines()
    minus = [x for x in full_corpus if x not in train_corpus]
    for example in test_dataset:
        #acc_mark = 0
        #print(example)
       
        first_token = example["input_ids"][0]
        second_token = example["input_ids"][1]
        
        #start_token = tokenizer.bos_token_id

        
        input_ids = torch.tensor([[0,first_token,second_token]]).to(model.device)
        #input_ids = torch.tensor([[0]]).to(device)
        for gen_len in range(5,20):
            with torch.no_grad():
                #is_complete_sentence = False
                #gen_len = 5
                
                #print("Model device:", model.device)
                generated_ids = model.generate(
                    input_ids,
                    max_length=gen_len,  
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(input_ids),
                    return_dict_in_generate=True,
                    temperature=temp, 
                    do_sample=True  
                )

      
            generated_sequence = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True)
            #print(input_ids)
           
            new_mark = 0
            
            if generated_sequence in full_corpus:
                #print(input_ids)
                #print(generated_sequence)
                correct_count += 1
                cor.append(generated_sequence)
                #acc_mark = 1
                if generated_sequence in minus:
                    new += 1
                    new_mark = 1
                    cor_new.append(generated_sequence) 
                    #print(generated_sequence)
                if generated_sequence in NP:
                    np_generated.append(generated_sequence)
                if generated_sequence in NPVP:
                    npvp_generated.append(generated_sequence)
                if generated_sequence in SOV:
                    sov_generated.append(generated_sequence)
                break
                
        if new_mark == 0 and generated_sequence in full_corpus:
            generated_ids = model.generate(
                    input_ids,
                    max_length= gen_len + 1, 
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                    attention_mask=torch.ones_like(input_ids),
                    return_dict_in_generate=True,
                    temperature=temp,  
                    do_sample=True
                )
            generated_sequence = tokenizer.decode(generated_ids.sequences[0], skip_special_tokens=True)
            if generated_sequence in minus:
                    new += 1
                    new_mark = 1 
                    cor.append(generated_sequence)
                    cor_new.append(generated_sequence)
                    #print(generated_sequence)
            if generated_sequence in NP:
                np_generated.append(generated_sequence)
            if generated_sequence in NPVP:
                npvp_generated.append(generated_sequence)
            if generated_sequence in SOV:
                sov_generated.append(generated_sequence)
        total_generated += 1
        if generated_sequence in full_corpus:
            #print(generated_sequence)
            cor.append(generated_sequence)
        if generated_sequence in minus:
            #print(generated_sequence)
            cor_new.append(generated_sequence)
        if generated_sequence in NP:
            np_generated.append(generated_sequence)
        if generated_sequence in NPVP:
            npvp_generated.append(generated_sequence)
        if generated_sequence in SOV:
            sov_generated.append(generated_sequence)
    ## if acc_mark ==1 !!!!!
    
    #print(generated_sequence)
    new_acc = new / ( new + (total_generated - correct_count) )
    accuracy = correct_count / total_generated
    new_generated = len(set(cor)) / total_generated
    new_no_re = len(set(cor_new)) / len(set(cor))
    np_per = len(set(np_generated)) / len(set(cor))#why not len(NP)
    npvp_per = len(set(npvp_generated)) / len(set(cor))#why not len(NPVP)
    sov_per = len(set(sov_generated)) / len(set(cor))#why not len(SOV)

    return (accuracy, new_acc, new_generated, new_no_re, np_per, npvp_per, sov_per)

## hyperparameter

In [ ]:
#hyperparams
layers = 2
#per_for_train = '10'
per_train_list = ['_human']
temp=1
lr = 5e-4#5e-5
train_epochs = 200#500
highest_accuracy = 0.90 ##the highest accuracy
labels_ids = {'true': 0, 'false': 1}
n_labels = len(labels_ids)
max_length = 25
#epochs = 500
batch_size = 20

evaluation_results = [[] for _ in range(len(per_train_list))]
evaluation_new_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_gg = [[] for _ in range(len(per_train_list))]
evaluation_np_per = [[] for _ in range(len(per_train_list))]
evaluation_npvp_per = [[] for _ in range(len(per_train_list))]
evaluation_sov_per = [[] for _ in range(len(per_train_list))]

In [ ]:
# Initialize the tokenizer
tokenizer = BPE_token()
save_path = 'tokenized_data'
# Load the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(save_path)
tokenizer.add_special_tokens({
    "eos_token": "</s>",
    "bos_token": "<s>",
    "unk_token": "<unk>",
    "pad_token": "</s>",###</s>
    "mask_token": "<mask>"
})
tokenizer.padding_side='left' # because it's gpt

## training

In [ ]:
evaluation_results = [[] for _ in range(len(per_train_list))]
evaluation_new_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_results = [[] for _ in range(len(per_train_list))]
evaluation_new_hit_gg = [[] for _ in range(len(per_train_list))]
evaluation_np_per = [[] for _ in range(len(per_train_list))]
evaluation_npvp_per = [[] for _ in range(len(per_train_list))]
evaluation_sov_per = [[] for _ in range(len(per_train_list))]
for index,per_for_train in enumerate(per_train_list):
    
    # Create configuration
    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        n_layer=layers ,
    )

    # Create the model and move it to GPU
    model = GPT2LMHeadModel(config).to(device)

    # Set the attention mask and pad token id
    model.config.pad_token_id = tokenizer.eos_token_id


    # Read test data
    test_file = "./text/all_sentences.txt" 
    test_dataset_g = LineByLineTextDataset(
        tokenizer=tokenizer,
        file_path=test_file,
        block_size=128  # Adjust block_size as per your requirement
    )
    linear_net = nn.Linear(in_features=768, out_features=2, bias=True)
    linear_net = linear_net.to(device)

    optimizer = torch.optim.AdamW([
                    {'params': model.parameters()},
                    {'params': linear_net.parameters()}
                ], lr=lr)
    # may also try this optimizer, because we just want a linear readout
    #optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Train and test
    acc_track = 0
    acc_track1 = 0
    acc_track2 = 0
    cla_acc_list = []
    loss_weight = []
    epochs_record = []
    for epoch in range(train_epochs):
        epoch_loss_weight = []
        loss_all = torch.zeros(1).to(device)
        lm_loss = torch.zeros(1).to(device)

        for cor_id, corpus in enumerate(train_list):
            output = model(torch.tensor([tokenizer.encode(corpus)]).to(model.device),
                        labels=torch.tensor([tokenizer.encode(corpus)]).to(device),
                           output_hidden_states=True)
            model_label = linear_net(output.hidden_states[-1][-1][-1])
            loss_c = criterion(model_label.unsqueeze(0), train_labels[cor_id].unsqueeze(0))  # 为批量维度添加维度
            loss_all += loss_c
            if corpus in new_list_co:
                lm_loss += output.loss
            if ((cor_id+1) % 50) == 0:
                #lm_weight = 10
                if epoch > 0 and new_rate > cla_accuracy:
                       lm_weight = 2
                else:
                    lm_weight = np.log( 79 / 2 )
                #if float(loss_all/lm_loss) < 1:
                #    lm_weight = 2
                #else:
                #    lm_weight = float(loss_all/lm_loss) * 2
                combined_loss = lm_weight * lm_loss + loss_all
                epoch_loss_weight.append(lm_weight)
                optimizer.zero_grad()   
                combined_loss.backward()  
                optimizer.step()         
                loss_all = torch.zeros(1).to(device)  
                lm_loss = torch.zeros(1).to(device)
        loss_weight.append(sum(epoch_loss_weight)/len(epoch_loss_weight))


        if epoch != 222:
            epochs_record.append(epoch)
            with torch.no_grad():
                correct_predictions = 0
                total_predictions = 0
                for cor_id, corpus in enumerate(val_list):
                    
                    output = model(torch.tensor([tokenizer.encode(corpus)]).to(model.device), output_hidden_states=True)
                    model_label = linear_net(output.hidden_states[-1][-1][-1])

                   
                    _, predicted_label = torch.max(model_label, dim=0)
                    true_label = val_labels[cor_id].item()

                    
                    if predicted_label.item() == true_label:
                        correct_predictions += 1
                    total_predictions += 1

           
            cla_accuracy = correct_predictions / total_predictions if total_predictions else 0
            cla_acc_list.append(cla_accuracy)
            print(f"Epoch {epoch + 1} Accuracy: {cla_accuracy:.2f}")
            
            reresult = evaluate_np_npvp_sov(model, tokenizer, test_dataset_g,per_for_train,temp)
            accuracy = reresult[0]
            new_rate = reresult[1]
            new_hit = reresult[2]
            new_gp = reresult[3]
            np_per = reresult[4]
            npvp_per = reresult[5]
            sov_per = reresult[6]
            evaluation_results[index].append(accuracy)
            evaluation_new_results[index].append(new_rate)
            evaluation_new_hit_results[index].append(new_hit)
            evaluation_new_hit_gg[index].append(new_gp)
            evaluation_np_per[index].append(np_per)
            evaluation_npvp_per[index].append(npvp_per)
            evaluation_sov_per[index].append(sov_per)
            
            output_dir = f"./pf_model/fine_tuned_epoch_{epoch}_new_rate_{new_rate}_cla_{cla_accuracy}_model"#layer=4_lr=1e-6__huaman%corpus
            model.save_pretrained(output_dir)
            
            print(f"Accuracy in {per_for_train}%corpus after {epoch+1} epochs: {accuracy:.2f}")
            print(f"New_rate in {per_for_train}%corpus after {epoch+1} epochs: {new_rate:.2f}")
            print(f"New_generated  in {per_for_train}%corpus after {epoch+1} epochs: {new_hit:.2f}")
            print(f"New_generated_without_repete  in {per_for_train}%corpus after {epoch+1} epochs: {new_gp:.2f}")
            print(f"np_per in {per_for_train}%corpus after {epoch+1} epochs: {np_per:.2f}")
            print(f"npvp_per in {per_for_train}%corpus after {epoch+1} epochs: {npvp_per:.2f}")
            print(f"sov_per in {per_for_train}%corpus after {epoch+1} epochs: {sov_per:.2f}")



## ploting

In [ ]:
gc_cla_acc_list = []
gc_new_rate_list = []
gc_models_list = []
gc_acc_list = []
gc_models_path = './pf_model/'
for name_id in range(200):
        for name in os.listdir(gc_models_path):
            if name[0:3] == 'fin' and name.split('_')[3] == str(name_id):# and (not (float(name.split('_')[5]) in cla_acc_list)):
                #print(name)
                gc_models_list.append(gc_models_path+name)
                gc_cla_acc_list.append(float(name.split('_')[8]))
                gc_new_rate_list.append(float(name.split('_')[6]))
                gc_acc_list.append(((float(name.split('_')[8]) + float(name.split('_')[6])) / 2))
        

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
epochs = range(200)

plt.figure(figsize=(8, 8))


original_new_results = gc_new_rate_list#evaluation_new_results[i]
smoothed_new_results = gaussian_filter1d(original_new_results, sigma=2)
plt.plot(epochs, smoothed_new_results, '#075B72', label='New Rate', linewidth=5)


original_cla_acc = gc_cla_acc_list
smoothed_cla_acc = gaussian_filter1d(original_cla_acc, sigma=2)
plt.plot(epochs, smoothed_cla_acc, '#30309A', label='Classification  Accuracy', linewidth=5)


max_new_y = max(smoothed_new_results ) 
max_new_x = epochs[original_new_results.index(max(original_new_results))]
plt.axhline(y=max_new_y, color='#075B72', linestyle='--', linewidth=1.5, alpha=0.8)
max_new_y_value = max(original_new_results )
plt.text(max_new_x - 3.5, 0.8, f'Max: {max_new_y_value:.2f}', fontsize=27, color='#075B72')


max_cla_y = max(smoothed_cla_acc)
max_cla_x = epochs[original_cla_acc.index(max(original_cla_acc))]
plt.axhline(y=max_cla_y, color='#30309A', linestyle='--', linewidth=1.5, alpha=0.8)
max_cla_y_value = max(original_cla_acc)
plt.text(100, max_cla_y , f'Max: {max_cla_y_value:.2f}', fontsize=27, color='#30309A')


plt.ylim(0, 1.0) 
plt.yticks(fontsize=25)
plt.xticks(fontsize=25)
plt.xlabel('Epochs', fontsize=25)
plt.ylabel('Accuracy', fontsize=25)
plt.grid(True)
plt.legend(loc='lower right', fontsize=20)


plt.show()
